# 04 — Facilities Bed Occupancy Silver

Builds `public_health.silver.facilities_bed_occupancy_silver` — anonymous hospital bed occupancy records.

### Source
- `public_health.bronze.facilities_bed_occupancy_bronze` (1,977 rows, 6 sheets: 2020–2025)

### Plan (facilities-silver-plan v1.4, section 3)
- Grain: 1 row per anonymous hospital record (no facility matching)
- Key: `(_source_key, _source_file, _source_sheet, _source_row_num)` = lineage columns
- Write: Rebuild (CREATE OR REPLACE)
- All rows loaded including ineligible ones (eligibility is a flag, not a disposition)
- 2025 sheet: 1,162 rows (418 Gov + 744 Pvt), province level, with bed counts
- 2020–2024: 815 rows, Private only, no bed counts → `private_only_year` exclusion
- BOR thresholds: 0–100 normal, >100–150 WARN (eligible), >150 flagged (ineligible)
- `bor_bed_basis = 'authorized'`, `bor_basis_confirmed = false` (D6 pending)
- `reporting_period_start/end` = Jan 1 – Dec 31 of sheet year, `period_confirmed = false`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid

BRONZE_TABLE = "public_health.bronze.facilities_bed_occupancy_bronze"
METADATA_TABLE = "public_health.bronze.ingestion_metadata"
TARGET_TABLE = "public_health.silver.facilities_bed_occupancy_silver"
SILVER_BUILD_ID = str(uuid.uuid4())

print(f"Bronze table : {BRONZE_TABLE}")
print(f"Target table : {TARGET_TABLE}")
print(f"Silver build ID: {SILVER_BUILD_ID}")

In [0]:
# Load bronze, cast numerics, parse sheet year
bronze = spark.table(BRONZE_TABLE)

# Cast BOR and bed counts to numeric
silver_df = (
    bronze
    .withColumn("bor", F.col("Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds").cast("double"))
    .withColumn("authorized_beds", 
        F.when(F.col("Authorized_Bed_Capacity").isNull() | (F.col("Authorized_Bed_Capacity") == ""), 
               F.lit(None))
        .otherwise(F.col("Authorized_Bed_Capacity").cast("int"))
    )
    .withColumn("implementing_beds",
        F.when(F.col("Implementing_Beds").isNull() | (F.col("Implementing_Beds") == ""),
               F.lit(None))
        .otherwise(F.col("Implementing_Beds").cast("int"))
    )
    .withColumn("reporting_year", F.col("_source_sheet").cast("int"))
)

total_rows = silver_df.count()
print(f"Total bronze rows: {total_rows}")
print(f"Rows with BOR: {silver_df.filter(F.col('bor').isNotNull()).count()}")
print(f"Rows with authorized_beds: {silver_df.filter(F.col('authorized_beds').isNotNull()).count()}")
print(f"Rows with implementing_beds: {silver_df.filter(F.col('implementing_beds').isNotNull()).count()}")

In [0]:
# Identify exact duplicates (same business columns) - plan says 1 group in 2025 sheet only
# Earlier sheets: identical rows stay separate (they can be different hospitals, never de-duplicated)
from pyspark.sql.window import Window

dup_cols = ["Region", "Province/District", "City/Municipality", "General",
            "Nature_of_Ownership", "Bed_Occupancy_Rate_BOR_Based_on_Authorized_Beds",
            "Authorized_Bed_Capacity", "Implementing_Beds"]

# Only check duplicates within 2025 sheet (the only sheet with bed counts)
dup_window = Window.partitionBy(*dup_cols).orderBy("_source_row_num")

silver_df = (
    silver_df
    .withColumn("_dup_rank", F.row_number().over(dup_window))
    .withColumn("_dup_count", F.count("*").over(Window.partitionBy(*dup_cols)))
    .withColumn(
        "is_exact_duplicate",
        F.when(
            (F.col("reporting_year") == 2025)
            & (F.col("_dup_count") > 1)
            & (F.col("_dup_rank") > 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .drop("_dup_rank", "_dup_count")
)

dup_count = silver_df.filter(F.col("is_exact_duplicate")).count()
print(f"Exact duplicate rows flagged (2025 only): {dup_count}")

In [0]:
# Apply eligibility rules per plan section 3.2-3.3
# Ineligible reasons: bor_null, bor_negative, bor_gt_150, level_missing, 
#   exact_duplicate, private_only_year (2020-2024)

silver_df = (
    silver_df
    .withColumn(
        "exclusion_reason",
        F.when(F.col("is_exact_duplicate"), F.lit("exact_duplicate"))
        .when(F.col("reporting_year") < 2025, F.lit("private_only_year"))
        .when(F.col("bor").isNull(), F.lit("bor_null"))
        .when(F.col("bor") < 0, F.lit("bor_negative"))
        .when(F.col("bor") > 150, F.lit("bor_gt_150"))
        .when(F.col("General").isNull() | (F.col("General") == ""), F.lit("level_missing"))
        .otherwise(F.lit(None))
    )
    .withColumn(
        "baseline_eligible",
        F.when(F.col("exclusion_reason").isNull(), F.lit(True))
        .otherwise(F.lit(False))
    )
)

# Summary
print("Eligibility summary:")
silver_df.groupBy("baseline_eligible", "exclusion_reason") \
    .agg(F.count("*").alias("rows")) \
    .orderBy(F.desc("baseline_eligible"), "exclusion_reason") \
    .show(20, truncate=False)

eligible_2025 = silver_df.filter((F.col("baseline_eligible")) & (F.col("reporting_year") == 2025)).count()
print(f"\nBaseline-eligible 2025 rows: {eligible_2025}")

In [0]:
# Add BOR basis, reporting period, and confirmed flags
silver_df = (
    silver_df
    .withColumn("bor_bed_basis", F.lit("authorized"))
    .withColumn("bor_basis_confirmed", F.lit(False))
    .withColumn(
        "reporting_period_start",
        F.to_date(F.concat(F.col("reporting_year").cast("string"), F.lit("-01-01")))
    )
    .withColumn(
        "reporting_period_end",
        F.to_date(F.concat(F.col("reporting_year").cast("string"), F.lit("-12-31")))
    )
    .withColumn("period_confirmed", F.lit(False))
)

print("Reporting period sample:")
silver_df.select("reporting_year", "reporting_period_start", "reporting_period_end").distinct().orderBy("reporting_year").show()

In [0]:
# Join ingestion_metadata for lineage: _load_batch_id and _content_checksum
# One metadata row per source_sheet (sheet per year in the workbook)
# Deduplicate: metadata may have duplicate entries from repeated loads - take latest retrieved_at
from pyspark.sql.window import Window

meta = spark.table(METADATA_TABLE).filter(F.col("source_name") == "bed_occupancy")

# Keep only the latest metadata row per source_sheet
meta_window = Window.partitionBy("source_sheet").orderBy(F.desc("retrieved_at"))
meta_latest = (
    meta
    .withColumn("_meta_rank", F.row_number().over(meta_window))
    .filter(F.col("_meta_rank") == 1)
    .drop("_meta_rank")
)

meta_lookup = meta_latest.select(
    F.col("source_sheet").alias("meta_sheet"),
    F.col("batch_id").alias("_load_batch_id"),
    F.col("content_checksum").alias("_content_checksum"),
)

print(f"Metadata rows before dedup: {meta.count()}")
print(f"Metadata rows after dedup: {meta_latest.count()}")

silver_df = (
    silver_df
    .join(
        meta_lookup,
        F.col("_source_sheet") == F.col("meta_sheet"),
        how="left"
    )
    .drop("meta_sheet")
)

# Check: every row should have lineage and no row duplication
null_lineage = silver_df.filter(F.col("_load_batch_id").isNull() | F.col("_content_checksum").isNull()).count()
row_count = silver_df.count()
print(f"Rows with missing lineage: {null_lineage}")
print(f"Row count after join: {row_count} (expected 1,977)")
assert null_lineage == 0, f"{null_lineage} rows missing lineage lookup"
assert row_count == 1977, f"Row count {row_count} != 1,977 - join is producing duplicates"
print("Lineage lookup complete: all rows have _load_batch_id and _content_checksum")

In [0]:
# Select final columns and write the table
final_df = (
    silver_df
    .select(
        # Business columns
        F.col("reporting_year"),
        F.col("Region").alias("region"),
        F.col("Province/District").alias("province_district"),
        F.col("City/Municipality").alias("city_municipality"),
        F.col("General").alias("hospital_level"),
        F.col("Nature_of_Ownership").alias("ownership"),
        F.col("bor"),
        F.col("authorized_beds"),
        F.col("implementing_beds"),
        F.col("bor_bed_basis"),
        F.col("bor_basis_confirmed"),
        F.col("reporting_period_start"),
        F.col("reporting_period_end"),
        F.col("period_confirmed"),
        F.col("baseline_eligible"),
        F.col("exclusion_reason"),
        F.col("is_exact_duplicate"),
        # Lineage columns (single-source set, no _metric_source_columns)
        F.lit(SILVER_BUILD_ID).alias("_silver_build_id"),
        F.col("_source_key"),
        F.col("_source_file"),
        F.col("_source_sheet"),
        F.col("_source_row_num"),
        F.col("_as_of_date"),
        F.col("_ingested_at"),
        F.col("_load_batch_id"),
        F.col("_content_checksum"),
        F.current_timestamp().alias("_silver_loaded_at"),
    )
)

print(f"Final row count: {final_df.count()}")
print(f"Columns: {len(final_df.columns)}")

# Write: Rebuild (CREATE OR REPLACE)
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)
print(f"Table written: {TARGET_TABLE}")

In [0]:
# Validation
result = spark.table(TARGET_TABLE)
total = result.count()
print(f"=== Validation: {TARGET_TABLE} ===")
print(f"Total rows: {total} (expected 1,977)")

# Key uniqueness
key_cols = ["_source_key", "_source_file", "_source_sheet", "_source_row_num"]
dup_keys = result.groupBy(*key_cols).agg(F.count("*").alias("cnt")).filter(F.col("cnt") > 1).count()
print(f"Duplicate keys: {dup_keys} (expected 0)")

# Null keys
null_keys = result.filter(
    F.col("_source_key").isNull() | F.col("_source_file").isNull() | 
    F.col("_source_sheet").isNull() | F.col("_source_row_num").isNull()
).count()
print(f"Null keys: {null_keys} (expected 0)")

# Sheet breakdown
print("\nSheet breakdown:")
result.groupBy("_source_sheet", "ownership") \
    .agg(F.count("*").alias("rows"), F.sum(F.col("baseline_eligible").cast("int")).alias("eligible")) \
    .orderBy("_source_sheet", "ownership") \
    .show()

# Eligibility summary
print("Eligibility summary:")
result.groupBy("baseline_eligible", "exclusion_reason") \
    .agg(F.count("*").alias("rows")) \
    .orderBy(F.desc("baseline_eligible"), "exclusion_reason") \
    .show(20, truncate=False)

# BOR stats for 2025 eligible rows
print("BOR stats for 2025 eligible rows:")
result.filter((F.col("baseline_eligible")) & (F.col("reporting_year") == 2025)) \
    .select(F.col("bor")).describe().show()

# Lineage completeness
lineage_cols = ["_silver_build_id", "_load_batch_id", "_content_checksum", "_silver_loaded_at"]
for c in lineage_cols:
    null_ct = result.filter(F.col(c).isNull()).count()
    print(f"  {c}: {null_ct} nulls")

assert total == 1977, f"Expected 1,977 rows, got {total}"
assert dup_keys == 0, f"{dup_keys} duplicate keys"
assert null_keys == 0, f"{null_keys} null keys"
print("\n✅ All validation checks passed")